# getitrack: Real-Video Multi-Object Tracking

`getitrack` is a lightweight multi-object tracking toolkit. This notebook runs the
**ByteTrack** algorithm on real videos, using a **getitune RF-DETR** detector via
getitrack's `GetiAdapter`.

Pipeline: **frame -> RF-DETR (getitune) -> `Detections` -> ByteTrack -> annotated frame**.
Classes to follow are selected with `ByteTrackConfig.class_filter`.

> Run this notebook in an environment that has both `getitune` and `getitrack` installed.

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd()))  # import the demo module next to this notebook

from IPython.display import Video, display

import getitrack
import getitune_demo
from getitune_demo import CLASS_SETS, build_detector, track_video

ROOT = Path(getitune_demo.__file__).resolve().parents[1]  # getitrack package root
getitrack.__version__

## Build the detector

`build_detector` loads a getitune RF-DETR model and wraps it with getitrack's
`GetiAdapter`, which exposes `detect(frame, frame_id) -> Detections`. Built once and
reused across videos.

In [ ]:
adapter = build_detector()
CLASS_SETS  # bikes -> bicycle/motorcycle, jets -> airplane, apples -> apple

## Run tracking live

Detect and track over the apples clip, then show the annotated result inline.
`ByteTrackConfig.class_filter` keeps only the apple detections.

In [ ]:
video = track_video(
    ROOT / 'videos' / 'apples.mp4',
    CLASS_SETS['apples'],
    ROOT / 'results' / 'apples_live.mp4',
    adapter,
    max_frames=125,
)
Video(str(video), embed=True, width=720)

## Pre-rendered results

The three demo videos tracked end to end. Colored boxes with ids are the tracker
output; each object keeps a stable id across the clip.

In [ ]:
for name in ['bikes', 'jets', 'apples']:
    path = ROOT / 'results' / f'{name}_tracked.mp4'
    if path.exists():
        print(name)
        display(Video(str(path), embed=True, width=720))

## Track your own video

Point `track_video` at any file and pass the COCO class ids to follow
(see `getitrack.utils.COCO_CLASSES` for the full list).

In [ ]:
# track_video(ROOT / 'videos' / 'vehicles.mp4', [3, 6, 8],
#             ROOT / 'results' / 'vehicles_tracked.mp4', adapter)
# Video(str(ROOT / 'results' / 'vehicles_tracked.mp4'), embed=True, width=720)